## **AI And Biotechnology/Bioinformatics**

## **AI and Drug Discovery Course: QSAR Modeling**
This notebook demonstrates how to collect and preprocess bioactivity data from ChEMBL for QSAR modeling

# **Part 2: Exploratory Data Analysis**

**Exploratory Data Analysis (EDA)** is an essential first step in data science and analysis. It is the process of investigating a dataset before modeling to discover patterns, spot anomalies, check assumptions, and understand the underlying structure of the data.

**The Main Goals of EDA**

When you perform EDA, you are fundamentally trying to answer a few core questions:


*   What is the shape and size of the data? (How many records and features do you have?)

*   Are there missing or corrupted values? (Do you need to drop or fill empty cells?)

*   How are the variables distributed? (Are the values tightly clustered, completely random, or spread across massive scales?)
*  Are there relationships or correlations? (Does changing one feature impact another?)





## **Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='ticks')

## **Import Bioactivity Dataset**

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df4 = pd.read_csv("bioactivity_curated_4ey77.csv")
df4.head(10)

## **Remove intermediate Values**

In [ ]:
print("Original shape:", df4.shape)

df4 = df4.dropna(subset=[
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value"
])

# convert IC50 to numeric
df4["standard_value"] = pd.to_numeric(df4["standard_value"], errors="coerce")

df4 = df4.dropna(subset=["standard_value"])

df4 = df4[df4["class"] != 'intermediate']

print("After cleaning:", df4.shape)


In [ ]:
df4 = df4[df4["class"] != 'intermediate']

In [ ]:
df4.head()

## **Aggregate Duplicates IC50 (median IC50 per canonical smile)**

In [ ]:
df_clean = (
    df4
    .groupby("canonical_smiles", as_index=False)
    .agg({
        "molecule_chembl_id": "first",
        "standard_value": "median",
        "class": "first" # Use 'class' instead of 'bioactivity_class'
    })
)

# Rename the 'class' column to 'bioactivity_class' for consistency with subsequent steps
df_clean = df_clean.rename(columns={'class': 'bioactivity_class'})

print("Before aggregation:", df4.shape[0])
print("After aggregation:", df_clean.shape[0])

df_clean.head()

In [ ]:
df_clean.standard_value.describe()

## **Convert IC50 to pIC50**

Convert IC50 to the negative logarithmic scale which is essentially -log10(IC50). This conversion allows IC50 data to be more uniformly distributed.

With raw IC50, the value represents the concentration of a drug required to inhibit a biological target by 50%. Therefore:

**Low IC50** = High Potency (Takes very little drug to do the job).

**High IC50** = Low Potency (Takes a massive amount of drug to do the job)

This inverse relationship is counterintuitive when interpreting model predictions or plotting graphs.By applying the negative log ($pIC_{50}$), we flip the direction:

**Higher pIC50** = More potent compound.

**Lower pIC50**= Less potent compound.


In [ ]:
df_clean["pIC50"] = -np.log10(df_clean["standard_value"] * 1e-9)

df_clean.head()


## **Reassign Activity Labels Based on PIC50**

Based on pIC50  
Active >= 6  
Inactive < 6  

In [ ]:
threshold = 6

df_clean["bioactivity_class"] = np.where(
    df_clean["pIC50"] >= threshold,
    "active",
    "inactive"
)

df_clean.head()


## **Check Duplicates**

In [ ]:
print("Duplicate SMILES remaining:",
      df_clean["canonical_smiles"].duplicated().sum())


In [ ]:
df_clean.pIC50.describe()

In [ ]:
finite_pic50 = df_clean[np.isfinite(df_clean["pIC50"])]

plt.hist(finite_pic50["pIC50"], bins=30)
plt.xlabel("pIC50")
plt.ylabel("Count")
plt.title("pIC50 Distribution")
plt.show()
plt.savefig('histogram_pic50.pdf')

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.countplot(x="bioactivity_class", data= df_clean, hue="bioactivity_class", edgecolor='black')

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('Frequency', fontsize=14, fontweight='bold')

# **Lipinski's Descriptor Calculation**

Christopher Lipinski, a scientist at Pfizer, came up with a set of rule-of-thumb for evaluating the druglikeness of compounds. Such druglikeness is based on the Absorption, Distribution, Metabolism and Excretion (ADME) that is also known as the pharmacokinetic profile. Lipinski analyzed all orally active FDA-approved drugs in the formulation of what is to be known as the Rule-of-Five or Lipinski's Rule.

The Lipinski's Rule stated the following:

**Molecular weight < 500 Dalton**  
**Octanol-water partition coefficient (LogP) < 5**  
**Hydrogen bond donors < 5**  
**Hydrogen bond acceptors < 10**

## **Install rdkit**

In [ ]:
!pip install rdkit

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from rdkit.Chem import rdMolDescriptors

In [ ]:
df_no_smiles = df_clean.drop(columns='canonical_smiles')

smiles = []

for i in df_clean.canonical_smiles.tolist():
  cpd = str(i).split('.')
  cpd_longest = max(cpd, key=len)
  smiles.append(cpd_longest)

smiles = pd.Series(smiles, name='canonical_smiles')

df_clean_smiles = pd.concat([df_no_smiles, smiles], axis=1)

df_clean_smiles


## **Calculate descriptors**

In [ ]:
def lipinski(smiles, verbose=False):

    moldata= []
    for elem in smiles:
        mol=Chem.MolFromSmiles(elem)
        moldata.append(mol)

    baseData= np.arange(1,1)
    i=0
    for mol in moldata:

        desc_MolWt = Descriptors.MolWt(mol)
        desc_MolLogP = Descriptors.MolLogP(mol)
        desc_NumHDonors = Lipinski.NumHDonors(mol)
        desc_NumHAcceptors = Lipinski.NumHAcceptors(mol)

        row = np.array([desc_MolWt,
                        desc_MolLogP,
                        desc_NumHDonors,
                        desc_NumHAcceptors])

        if(i==0):
            baseData=row
        else:
            baseData=np.vstack([baseData, row])
        i=i+1

    columnNames=["MW","LogP","NumHDonors","NumHAcceptors"]
    descriptors = pd.DataFrame(data=baseData,columns=columnNames)

    return descriptors



In [ ]:
df_lipinski = lipinski(df_clean_smiles.canonical_smiles)
df_lipinski.head()

In [ ]:
df_lipinski.shape

Combine Both datasets

In [ ]:
df_clean_smiles.head()

In [ ]:
df_combined = pd.concat([df_clean_smiles, df_lipinski], axis=1)
df_combined.head()

In [ ]:
df_combined = df_combined.drop(columns="standard_value")
df_combined.head()

In [ ]:
# Save CSV
df_combined.to_csv("df_lipinski.csv", index=False)

In [ ]:
from google.colab import files
files.download("df_lipinski.csv")

## **Exploratory Data Analysis or Chemical Space Analysis For Lipinski Descriptors**

## **Barplot of the bioactivity classes**

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.countplot(x="bioactivity_class", data= df_combined, hue="bioactivity_class", edgecolor='black')

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('Frequency', fontsize=14, fontweight='bold')

plt.savefig('barplot_bioactivity_class.pdf')

## **Boxplot of the bioactivity classes for PIC50**

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.boxplot(x = "bioactivity_class", y = "pIC50", data = df_combined, hue="bioactivity_class")

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('pIC50 value', fontsize=14, fontweight='bold')

## **Scatter of Molecular Weight vs Solubility (LogP)**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(5.5, 5.5))

# Filter out non-finite values from all relevant plotting columns (MW, LogP, pIC50)
df_plot = df_combined[np.isfinite(df_combined['pIC50']) &
                      np.isfinite(df_combined['MW']) &
                      np.isfinite(df_combined['LogP'])]

sns.scatterplot(x='MW', y='LogP', data=df_plot, hue='bioactivity_class', size='pIC50', edgecolor='black', alpha=0.7)

plt.xlabel('MW', fontsize=14, fontweight='bold')
plt.ylabel('LogP', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0)
plt.savefig('scatter_plot_MW_vs_LogP.pdf', bbox_inches='tight')

In [ ]:
 import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, ax = plt.subplots(figsize=(5.5, 5.5))

# Filter out non-finite values from all relevant plotting columns (MW, LogP, pIC50)
df_plot = df_combined[np.isfinite(df_combined['pIC50']) &
                      np.isfinite(df_combined['MW']) &
                      np.isfinite(df_combined['LogP'])]

sns.scatterplot(x='MW', y='LogP', data=df_plot, hue='bioactivity_class', size='pIC50', edgecolor='black', alpha=0.7,
                palette={'active': '#2E7D32', 'inactive': '#C62828', 'intermediate': '#EF6C00'},
                s=100, ax=ax)
ax.axvline(500, color='gray', linestyle=':')
ax.axhline(5, color='gray', linestyle=':')

plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0)
plt.savefig('scatter_plot_MW_vs_LogP.pdf', bbox_inches='tight')

In [ ]:
# Correlation heatmap of key Lipinski descriptors + pIC50
plt.figure(figsize=(6, 5))
corr = df_combined[["MW", "LogP", "NumHDonors", "NumHAcceptors", "pIC50"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.show()

# **Lecture 3**

## **Statistical analysis (Mann-Whitney U Test)**

In [ ]:
def mannwhitney(descriptor, df_combined, verbose=False):
    """
    Perform Mann-Whitney U test between active and inactive compounds
    for a given descriptor.

    Parameters:
    - descriptor : str, column name of the descriptor
    - df_combined : pandas DataFrame, must have columns [descriptor, bioactivity_class]
    - verbose : bool, if True prints the test statistics

    Returns:
    - results : pandas DataFrame with test statistics, p-value, and interpretation
    """
    from numpy.random import seed
    from scipy.stats import mannwhitneyu
    import pandas as pd

    # set seed for reproducibility
    seed(1)

    # select only relevant columns
    df = df_combined[[descriptor, 'bioactivity_class']]

    # separate active and inactive compounds
    active = df[df['bioactivity_class'] == 'active'][descriptor]
    inactive = df[df['bioactivity_class'] == 'inactive'][descriptor]

    # perform Mann-Whitney U test
    stat, p = mannwhitneyu(active, inactive)

    if verbose:
        print(f"Descriptor: {descriptor}")
        print(f"Statistics={stat:.3f}, p={p:.3f}")

    # interpret result
    alpha = 0.05
    if p > alpha:
        interpretation = 'Same distribution (fail to reject H0)'
    else:
        interpretation = 'Different distribution (reject H0)'

    # store results in a DataFrame
    results = pd.DataFrame({
        'Descriptor': descriptor,
        'Statistics': stat,
        'p': p,
        'alpha': alpha,
        'Interpretation': interpretation
    }, index=[0])

    # save results to CSV
    filename = 'mannwhitneyu_' + descriptor + '.csv'
    results.to_csv(filename, index=False)

    return results


# **pIC50**

In [ ]:
mannwhitney("pIC50", df_combined, verbose=True)

# **Molecular Weight**



In [ ]:
mannwhitney("MW", df_combined, verbose=True)

# **Solubility LogP**

In [ ]:
mannwhitney("LogP", df_combined, verbose=True)

# **Number of Hydrogen Donors**

In [ ]:
mannwhitney("NumHDonors", df_combined, verbose=True)

# **Number of Hydrogen Acceptors**

In [ ]:
mannwhitney("NumHAcceptors", df_combined, verbose=True)

# **Combine All Statistical Results**

In [ ]:
import pandas as pd
import glob
import os

# Get list of all Mann-Whitney CSV files in current folder
mw_files = glob.glob("mannwhitneyu_*.csv")

# Combine them into one DataFrame
mw_combined = pd.concat([pd.read_csv(f) for f in mw_files], ignore_index=True)

# Save combined CSV
combined_filename = "mannwhitney_summary.csv"
mw_combined.to_csv(combined_filename, index=False)

print(f"Combined Mann-Whitney CSV saved as {combined_filename}")

In [ ]:
mw_combined

## **Molecular Weight**

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.boxplot(
    x='bioactivity_class',
    y='MW',
    data=df_combined,
    hue='bioactivity_class'

)

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('MW', fontsize=14, fontweight='bold')

plt.savefig('boxplot_MW.pdf')


## **logP**

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.boxplot(
    x="bioactivity_class",
    y='LogP',
    data=df_combined,
    hue="bioactivity_class"
)

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('LogP', fontsize=14, fontweight='bold')

plt.savefig('boxplot_LogP.pdf')

## **NumHDonors**

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.boxplot(
    x="bioactivity_class",
    y="NumHDonors",
    data=df_combined,
    hue="bioactivity_class",
    palette={
        "inactive": "red",
        "active": "yellow",
        "intermediate": "orange"
    }
)

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('NumHDonors', fontsize=14, fontweight='bold')

plt.savefig('boxplot_NumHDonors.pdf')

## **NumHAcceptors**

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.boxplot(
    x="bioactivity_class",
    y="NumHAcceptors",
    data=df_combined,
    hue="bioactivity_class"
)

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('NumHAcceptors', fontsize=14, fontweight='bold')

plt.savefig('boxplot_NumHAcceptors.pdf')

# **Some Extra Graphs (if you need):**

In [ ]:
# Distribution of Molecular Weight (MW)
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(data=df_combined, x='MW', kde=True, color='#1F4E79', ax=ax, bins=15)
ax.axvline(500, color='red', linestyle='--', label='Lipinski Limit (500 Da)')
ax.set_title('Molecular Weight Distribution', fontweight='bold', fontsize=12)
ax.set_xlabel('MW (g/mol)')
ax.legend()

In [ ]:
# Distribution of Biological Potency (pIC50)
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(data=df_combined, x='pIC50', kde=True, color='#7A2048', ax=ax, bins=10)
ax.set_title('Bioactivity Potency Distribution', fontweight='bold', fontsize=12)
ax.set_xlabel('pIC50 Values')

## **Save & Downlaod Results**

In [ ]:
! zip -r EDA_results.zip . -i *df_lipinski.csv *mannwhitney_summary.csv *.pdf

# **Lipinski filter** for selecting top compounds

In [ ]:
# Select active compounds only
active_compounds = df_combined[df_combined['bioactivity_class'] == 'active']

# Apply Lipinski filters
lipinski_pass = active_compounds[
    (active_compounds['MW'] <= 500) &
    (active_compounds['LogP'] <= 5) &
    (active_compounds['NumHDonors'] <= 5) &
    (active_compounds['NumHAcceptors'] <= 10)
]

# Rank by pIC50, highest activity first
top_docking_compounds = lipinski_pass.sort_values(
    by='pIC50',
    ascending=False
).head(20)

top_docking_compounds

In [ ]:
import pandas as pd
import os
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from google.colab import files

# Step 1: Select top 100 compounds
active_compounds = df_combined[df_combined['bioactivity_class'] == 'active'].copy()

lipinski_pass = active_compounds[
    (active_compounds['MW'] <= 500) &
    (active_compounds['LogP'] <= 5) &
    (active_compounds['NumHDonors'] <= 5) &
    (active_compounds['NumHAcceptors'] <= 10)
].copy()

top_100 = lipinski_pass.sort_values(by='pIC50', ascending=False).head(100).reset_index(drop=True)

print("Total selected compounds:", top_100.shape[0])
top_100.head()

# **Save Top 100 in excel filw with their structures**

In [ ]:
# Step 2: Create folders
os.makedirs("compound_structures", exist_ok=True)

# Step 3: Create workbook
wb = Workbook()
ws = wb.active
ws.title = "Top_100_Compounds"

# Excel headers
headers = [
    "Rank",
    "Molecule_ID",
    "Structure",
    "SMILES",
    "pIC50",
    "MW",
    "LogP",
    "NumHDonors",
    "NumHAcceptors"
]

ws.append(headers)

# Column width
ws.column_dimensions['A'].width = 8
ws.column_dimensions['B'].width = 20
ws.column_dimensions['C'].width = 28
ws.column_dimensions['D'].width = 60
ws.column_dimensions['E'].width = 12
ws.column_dimensions['F'].width = 12
ws.column_dimensions['G'].width = 12
ws.column_dimensions['H'].width = 15
ws.column_dimensions['I'].width = 18

# Step 4: Insert data and molecular structure image
for idx, row in top_100.iterrows():
    rank = idx + 1
    mol_id = row['molecule_chembl_id']
    smiles = row['canonical_smiles']

    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:
        img_path = f"compound_structures/{mol_id}.png"
        Draw.MolToFile(mol, img_path, size=(250, 200))

        ws.append([
            rank,
            mol_id,
            "",
            smiles,
            row['pIC50'],
            row['MW'],
            row['LogP'],
            row['NumHDonors'],
            row['NumHAcceptors']
        ])

        img = XLImage(img_path)
        img.width = 180
        img.height = 140

        cell_position = f"C{idx + 2}"
        ws.add_image(img, cell_position)

        ws.row_dimensions[idx + 2].height = 110

    else:
        ws.append([
            rank,
            mol_id,
            "Invalid SMILES",
            smiles,
            row['pIC50'],
            row['MW'],
            row['LogP'],
            row['NumHDonors'],
            row['NumHAcceptors']
        ])

# Step 5: Save Excel
excel_file = "Top_100_compounds_with_structures.xlsx"
wb.save(excel_file)

files.download(excel_file)

# **Save all molcules in sdf**

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

# Define the output path for your SDF file
sdf_file = "Top_100_compounds.sdf"

# Initialize the RDKit SDWriter
writer = Chem.SDWriter(sdf_file)

print("Starting SDF generation...")

for idx, row in top_100.iterrows():
    mol_id = row['molecule_chembl_id']
    smiles = row['canonical_smiles']

    # Generate the RDKit molecule object from SMILES
    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:
        # CRITICAL: Assign the ChEMBL ID as the internal molecule name
        mol.SetProp("_Name", str(mol_id))

        # Add the numeric descriptors as metadata properties within the SDF
        mol.SetProp("Rank", str(idx + 1))
        mol.SetProp("pIC50", f"{row['pIC50']:.3f}")
        mol.SetProp("MW", f"{row['MW']:.2f}")
        mol.SetProp("LogP", f"{row['LogP']:.2f}")
        mol.SetProp("NumHDonors", str(int(row['NumHDonors'])))
        mol.SetProp("NumHAcceptors", str(int(row['NumHAcceptors'])))

        # Generate 2D coordinates so the structures render beautifully in visualization tools
        AllChem.Compute2DCoords(mol)

        # Write the molecule to the file
        writer.write(mol)
    else:
        print(f"Skipping invalid SMILES for molecule: {mol_id}")

# Close the writer to finalize and save the file safely
writer.close()
print(f"Successfully saved SDF file as: {sdf_file}")

# Download the file to your local computer (Google Colab environment)
files.download(sdf_file)

# **Additional tables**

In [ ]:
import pandas as pd
from google.colab import files

descriptor_table = pd.DataFrame({
    "Descriptor": [
        "Molecular weight",
        "Octanol-water partition coefficient",
        "Hydrogen bond donors",
        "Hydrogen bond acceptors",
        "pIC50",
        "Bioactivity class"
    ],
    "Abbreviation": [
        "MW",
        "LogP",
        "NumHDonors",
        "NumHAcceptors",
        "pIC50",
        "active/inactive"
    ],
    "Descriptor type": [
        "Physicochemical descriptor",
        "Physicochemical descriptor",
        "Lipinski descriptor",
        "Lipinski descriptor",
        "Bioactivity descriptor",
        "Activity label"
    ],
    "Description": [
        "Molecular mass of the compound, used to assess drug-likeness and oral bioavailability.",
        "Estimates compound lipophilicity and membrane permeability.",
        "Number of hydrogen-bond donor groups present in the molecule.",
        "Number of hydrogen-bond acceptor groups present in the molecule.",
        "Negative logarithmic transformation of IC50, used as the dependent variable for QSAR modelling.",
        "Compounds were classified as active when pIC50 ≥ 6 and inactive when pIC50 < 6."
    ]
})

descriptor_table.to_excel("Descriptor_table_for_QSAR.xlsx", index=False)

descriptor_table

In [ ]:
files.download("Descriptor_table_for_QSAR.xlsx")

**Final check**

In [ ]:
print(df_combined['bioactivity_class'].value_counts())